# Ngay 25 26 28 CNN · Kaggle
Bật GPU nếu có. Notebook tự chọn CUDA, MPS hoặc CPU. Chấp nhận luật cuộc thi trước khi tải. Thực thi và nộp submission thủ công. Cấu hình ít epoch để thử quy trình.

Gắn dữ liệu qua Add Input. Fashion-MNIST tải qua torchvision khi phiên có Internet.

In [ ]:
from pathlib import Path
import os, random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps' if torch.backends.mps.is_available() else 'cpu')
def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
def train_epoch(model, loader, optimizer):
    model.train()
    for x,y in loader:
        optimizer.zero_grad()
        loss=nn.functional.cross_entropy(model(x.to(device)),y.to(device))
        loss.backward()
        optimizer.step()
@torch.inference_mode()
def predict(model, loader):
    model.eval()
    return torch.cat([model(batch[0].to(device)).argmax(1).cpu() for batch in loader])
print('Thiết bị:', device)

In [ ]:
ROOT=Path('/kaggle/working')
INPUT_DIR=Path('/kaggle/input/digit-recognizer')
# Khi Add Input tạo thư mục khác, sửa INPUT_DIR cho đúng.

In [ ]:
from torchvision.datasets import FashionMNIST
from torchvision.transforms import ToTensor

def conv2d_from_patches(x, weight, bias):
    k=weight.shape[-1]
    patches=nn.functional.unfold(x, kernel_size=k, padding=k//2)
    flat=weight.flatten(1) @ patches + bias[None,:,None]
    return flat.reshape(len(x),len(weight),x.shape[-2],x.shape[-1])

seed_all(17)
layer=nn.Conv2d(1,4,3,padding=1)
sample=torch.randn(2,1,28,28)
torch.testing.assert_close(conv2d_from_patches(sample,layer.weight,layer.bias),layer(sample))
base=FashionMNIST(ROOT/'fashion',train=True,download=True,transform=ToTensor())
fit,validation,_=random_split(base,[5000,1000,len(base)-6000],generator=torch.Generator().manual_seed(17))

def cnn(dropout):
    return nn.Sequential(nn.Conv2d(1,12,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
                         nn.Flatten(),nn.Dropout(dropout),nn.Linear(12*14*14,10)).to(device)

results=[]
for seed in (17,23,41):
    for dropout in (0.,.2):
        seed_all(seed)
        model=cnn(dropout)
        optimizer=torch.optim.Adam(model.parameters(),lr=.002,weight_decay=.0001)
        loader=DataLoader(fit,batch_size=64,shuffle=True,generator=torch.Generator().manual_seed(seed))
        for _ in range(3):
            train_epoch(model,loader,optimizer)
        prediction=predict(model,DataLoader(validation,batch_size=128))
        truth=torch.tensor([y for _,y in validation])
        results.append(dict(seed=seed,device=str(device),dropout=dropout,accuracy=(prediction==truth).float().mean().item()))
display(pd.DataFrame(results))

In [ ]:
# Tập Digit Recognizer độc lập với Fashion-MNIST: huấn luyện lại trước khi xuất tệp.
train=pd.read_csv(INPUT_DIR/'train.csv')
test=pd.read_csv(INPUT_DIR/'test.csv')
pixel_columns=list(train.columns.drop('label'))
def pixels(frame):
    return torch.tensor(frame[pixel_columns].to_numpy(dtype=np.float32)).reshape(-1,1,28,28)/255
seed_all(17)
model=cnn(.2)
optimizer=torch.optim.Adam(model.parameters(),lr=.002)
dataset=TensorDataset(pixels(train),torch.tensor(train.label.to_numpy(),dtype=torch.long))
for _ in range(3):
    train_epoch(model,DataLoader(dataset,batch_size=128,shuffle=True,generator=torch.Generator().manual_seed(17+_)),optimizer)
prediction=predict(model,DataLoader(TensorDataset(pixels(test)),batch_size=128))
submission=pd.DataFrame({'ImageId':np.arange(1,len(test)+1),'Label':prediction.numpy()})
submission.to_csv(ROOT/'digit_submission.csv',index=False)